In [12]:
import csv
import json
import os
import time
import pandas as pd
import numpy as np
import cv2
import gc

import torch
# 清空显存
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

from accelerate import Accelerator
from torch.utils.tensorboard import SummaryWriter

from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.utils import set_determinism

import sys

from src.data_utils import parse_scan_filename, build_case_day_slices, build_rle_index,load_case_days,load_val_vis_samples
from src.constants import CLASSES, CLASS2IDX
from src.rle import rle_encode, rle_decode
from src.datasets import LoadCaseDayVolumed

from monai.networks.nets import Unet
from monai.networks.layers import Norm

from accelerate import Accelerator
import logging
from pathlib import Path


## Config

In [13]:
class TrainCfg:
    #data
    data_root: str = './inputs'
    train_ids: str = 'inputs/splits/c1_train_case_days.csv'
    val_ids: str = 'inputs/splits/c1_val_case_days.csv'
    num_workers: int = 6

    # checkpointing
    resume_from: str = ''  # path to checkpoint to resume from, e.g.

    # model
    model_name: str = 'Unet3D_c1'
    spatial_dims: int = 3
    in_channels: int = 1
    out_channels: int = len(CLASSES)

    patch_d: int = 96
    patch_h: int = 224
    patch_w: int = 224
    patch_size: tuple = (patch_d, patch_h, patch_w)
    
    # optimizer
    lr: float = 1e-4
    weight_decay: float = 1e-4

    # OneCycleLR
    lr_max: float = 5e-4
    lr_pct_start: float = 0.2
    lr_div_factor: float = 25
    lr_final_div_factor: float = 1000
    # training
    epochs: int = 150
    val_interval: int = 3
    mixed_precision: str = 'fp16'
    train_batch_size: int = 16
    val_batch_size: int = 1

    # val
    sw_batch_size: int = 4
    sw_overlap: float = 0.25

    # TensorBoard val sample visualization
    val_vis_every: int = 5
    val_vis_samples: str = './inputs/splits/val_vis_samples.json'

    # caching
    train_cache_rate: float = 1
    val_cache_rate: float = 1

cfg = TrainCfg()

In [14]:
from monai.transforms import (
    AsDiscrete,
    AsDiscreted,
    EnsureChannelFirstd,
    EnsureTyped,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandCropd,
    RandSpatialCropd,
    RandCropByPosNegLabeld,
    SaveImaged,
    ScaleIntensityRanged,
    ScaleIntensityd,
    Spacingd,
    Invertd,
    MapTransform,
    SpatialPadd,
)
from monai.data import CacheDataset, DataLoader, Dataset
from monai.inferers import sliding_window_inference
from monai.data import pad_list_data_collate

import matplotlib.pyplot as plt

set_determinism(42)

In [15]:
def setup_logger(log_path: str):
    logger = logging.getLogger("train")
    logger.setLevel(logging.INFO)
    logger.propagate = False  # 防止重复输出

    # 防止重复添加 handler
    if logger.handlers:
        for handler in logger.handlers[:]:
            logger.removeHandler(handler)

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    # 控制台
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    # 文件
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger

# Create a unique output directory for this run
run_timestamp = time.strftime("%Y%m%d-%H%M%S")
output_dir = f"outputs/{run_timestamp}_{cfg.model_name}"
os.makedirs(output_dir, exist_ok=True)

# Save config to json
def save_config(cfg, path):
    cfg_dict = {k: v for k, v in cfg.__class__.__dict__.items() if not k.startswith('__')}
    # Convert non-serializable types
    for k, v in cfg_dict.items():
        if isinstance(v, tuple):
            cfg_dict[k] = list(v)
    
    with open(path, 'w') as f:
        json.dump(cfg_dict, f, indent=2)

save_config(cfg, os.path.join(output_dir, "config.json"))


logger = setup_logger(f"{output_dir}/train.log")
writer = SummaryWriter(log_dir=f"{output_dir}/tb")

## Build metainfo

In [16]:
# Build slices index
data_dir = os.path.join(cfg.data_root, 'train')
data_csv = os.path.join(cfg.data_root, 'train.csv')

case_day_slices = build_case_day_slices(data_dir)
all_case_days = sorted(case_day_slices.keys())
logger.info(f"Found case_day volumes: {len(all_case_days)}")

# Build rle index
rle_index = build_rle_index(data_csv)

train_set = set(load_case_days(cfg.train_ids))
val_set = set(load_case_days(cfg.val_ids))

overlap = sorted(train_set & val_set)
if overlap:
    raise ValueError(f"train_ids and val_ids overlap (n={len(overlap)}), e.g. {overlap[:5]}")

logger.info(f"Train case_days: {len(train_set)}, Val case_days: {len(val_set)}")

train_files_d = [{"case_day": cd} for cd in sorted(train_set)]
val_files_d = [{"case_day": cd} for cd in sorted(val_set)]

val_vis_samples = load_val_vis_samples(cfg.val_vis_samples)
logger.info(f"Validation visualization samples: {len(val_vis_samples['samples'])}")

2026-02-07 03:49:10,415 | INFO | Found case_day volumes: 274
2026-02-07 03:49:16,914 | INFO | Train case_days: 74, Val case_days: 18
2026-02-07 03:49:16,916 | INFO | Validation visualization samples: 16


## Build dataset

In [17]:
train_transforms = Compose(
    [
        LoadCaseDayVolumed(
            keys=["case_day"],
            case_day_slices=case_day_slices,
            rle_index=rle_index,
        ),
        EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
        ScaleIntensityd(
            keys=["image"],
            minv=0.0,
            maxv=1.0,
        ),
        SpatialPadd(
            keys=["image", "label"],
            spatial_size=(cfg.patch_d, cfg.patch_h, cfg.patch_w),
            mode="constant",
        ),
        RandSpatialCropd(
            keys=["image", "label"],
            roi_size=(cfg.patch_d, cfg.patch_h, cfg.patch_w),
            random_size=False,
        ),
        EnsureTyped(keys=["image", "label"]),
    ]
)

val_transforms = Compose(
    [
        LoadCaseDayVolumed(
            keys=["case_day"],
            case_day_slices=case_day_slices,
            rle_index=rle_index,
        ),
        EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
        ScaleIntensityd(
            keys=["image"],
            minv=0.0,
            maxv=1.0,
        ),
        SpatialPadd(
            keys=["image", "label"],
            spatial_size=(cfg.patch_d, cfg.patch_h, cfg.patch_w),
            mode="constant",
        ),
        EnsureTyped(keys=["image", "label"]),
    ]
)

### Datasets

In [18]:
cache_time_start = time.time()
logger.info("cache num_workers: %d", cfg.num_workers)
logger.info("Caching train dataset... (%d items)", len(train_files_d))
logger.info("Caching val dataset... (%d items)", len(val_files_d))
train_ds = CacheDataset(
    data=train_files_d,
    transform=train_transforms,
    cache_rate=cfg.train_cache_rate,
    num_workers=cfg.num_workers,
    progress=False,
)

val_ds = CacheDataset(
    data=val_files_d,
    transform=val_transforms,
    cache_rate=cfg.val_cache_rate,
    num_workers=cfg.num_workers,
    progress=False,
)

logger.info(f"Caching time: {time.time() - cache_time_start:.2f} seconds")



2026-02-07 03:49:16,960 | INFO | cache num_workers: 6
2026-02-07 03:49:16,961 | INFO | Caching train dataset... (74 items)
2026-02-07 03:49:16,962 | INFO | Caching val dataset... (18 items)


2026-02-07 03:49:24,924 | INFO | Caching time: 7.96 seconds


## If resume

In [19]:
# from src.checkpointing import load_resume_checkpoint

# chkpt = load_resume_checkpoint(cfg.resume_from)


## Model, Loss, Optimizer

In [20]:
accelerator = Accelerator(mixed_precision=cfg.mixed_precision)
    
model = Unet(
    spatial_dims=cfg.spatial_dims,
    in_channels=cfg.in_channels,
    out_channels=cfg.out_channels,
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    dropout=0.2,
    norm=Norm.BATCH,
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.train_batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=pad_list_data_collate,  # 处理不同尺寸的样本
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.val_batch_size,  # 验证时使用 batch_size=1 避免 collate 问题
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

loss_function = DiceCELoss(sigmoid=True, squared_pred=True, reduction="mean")

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
steps_per_epoch = len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=cfg.lr_max,
    epochs=cfg.epochs,
    steps_per_epoch=steps_per_epoch,
    pct_start=cfg.lr_pct_start, # 20% steps for warmup
    div_factor=cfg.lr_div_factor, # lr = max_lr/div_factor at the start of training
    final_div_factor=cfg.lr_final_div_factor, # lr = max_lr/final_div_factor at the end of training
)


dice_metric = DiceMetric(
    include_background=True,
    reduction="mean_batch",
    get_not_nans=False,
)


## Training

In [21]:
def train_one_epoch(model, train_loader, optimizer, loss_function, accelerator, scheduler, global_step):
    model.train()
    epoch_loss = 0.0
    epoch_grad_norm = 0.0
    device = accelerator.device
    for batch in train_loader:
        # prepare 后数据已经在正确设备上，但 dict batch 需要手动移动
        images = batch["image"].to(device)
        labels = batch["label"].to(device).float()

        optimizer.zero_grad()

        with accelerator.autocast():
            logits = model(images)
            loss = loss_function(logits, labels)

        accelerator.backward(loss)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float('inf'))
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        epoch_grad_norm += grad_norm.item()
    
    return epoch_loss / len(train_loader), epoch_grad_norm / len(train_loader), global_step + len(train_loader)


@torch.no_grad()
def compute_patch_dice(
    model,
    data_loader,
    accelerator,
    dice_metric,
    threshold: float = 0.5,
    include_background: bool = True,
):
    """
    Patch-level Dice on a loader (train or val).
    Returns: (dice_per_class_tensor, dice_mean_float)
    """
    model.eval()
    device = accelerator.device

    for batch in data_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device).float()

        with accelerator.autocast():
            logits = model(images)
        probs = torch.sigmoid(logits)
        preds = (probs > float(threshold)).float()

        dice_metric(y_pred=preds, y=labels)

    dice_per_class = dice_metric.aggregate()
    dice_mean = float(dice_per_class.mean().item())
    return dice_per_class, dice_mean

@torch.no_grad()
def compute_sw_dice(
    model,
    data_loader,
    accelerator,
    dice_metric,
    roi_size,
    sw_batch_size,
    sw_overlap,
    threshold: float = 0.5,
    include_background: bool = True,
):
    """
    Compute sliding window inference Dice on val set.
    Returns: (dice_per_class_tensor, dice_mean_float)
    """
    model.eval()
    device = accelerator.device
    total_loss = 0.0

    for batch in data_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device).float()

        # [B, C, D, H, W] -> [B, C, D, H, W]
        with accelerator.autocast():
            logits = sliding_window_inference(
                images,
                roi_size=roi_size,
                sw_batch_size=sw_batch_size,
                predictor=model,
                overlap=sw_overlap,
            )
            loss = loss_function(logits, labels)
        total_loss += loss.item()
        probs = torch.sigmoid(logits)
        preds = (probs > float(threshold)).float()

        dice_metric(y_pred=preds, y=labels)

    dice_per_class = dice_metric.aggregate()
    dice_mean = float(dice_per_class.mean().item())
    return dice_per_class, dice_mean, total_loss / len(data_loader)


In [22]:


# Accelerate prepare - 启用完整的混合精度训练（包含 GradScaler）
model, optimizer, train_loader, val_loader = accelerator.prepare(
    model, optimizer, train_loader, val_loader
)
# loss_function 需要手动移动到设备
loss_function = loss_function.to(accelerator.device)

logger.info(f"Mixed precision: {cfg.mixed_precision}, Device: {accelerator.device}")
logger.info(f"Scaler enabled: {accelerator.scaler is not None}")
logger.info(f"Output directory: {output_dir}")

# 预计算并记录完整的lr schedule到TensorBoard
total_steps = cfg.epochs * steps_per_epoch
temp_optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
temp_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    temp_optimizer,
    max_lr=1e-3,
    epochs=cfg.epochs,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.1,
    div_factor=10.0,
    final_div_factor=100.0,
)
for step in range(total_steps):
    lr = temp_optimizer.param_groups[0]['lr']
    writer.add_scalar("schedule/lr_planned", lr, step)
    temp_scheduler.step()
del temp_optimizer, temp_scheduler
logger.info(f"Logged planned lr schedule ({total_steps} steps) to TensorBoard")

start_epoch = 1
best_metric = -1.0
best_metric_epoch = -1


global_step = (start_epoch - 1) * max(1, len(train_loader))

training_start_time = time.time()
train_times = []
val_times = []

for epoch in range(start_epoch, cfg.epochs + 1):
    t0 = time.time()
    model.train()
    epoch_loss, epoch_grad_norm, global_step = train_one_epoch(
        model, train_loader, optimizer, loss_function, accelerator, scheduler, global_step
    )
    train_time = time.time() - t0
    train_times.append(train_time)

    current_lr = optimizer.param_groups[0]['lr']
    logger.info("Epoch %d loss=%.4f lr=%.2e train_time=%.1fs", epoch, epoch_loss, current_lr, train_time)
    writer.add_scalar("train/loss", float(epoch_loss), epoch)
    writer.add_scalar("train/lr", current_lr, epoch)
    writer.add_scalar("train/grad_norm", float(epoch_grad_norm), epoch)
    writer.add_scalar("system/train_time_sec", train_time, epoch)

    # 记录显存占用
    if torch.cuda.is_available():
        mem_alloc = torch.cuda.memory_allocated() / 1024**3  # GB
        mem_reserved = torch.cuda.memory_reserved() / 1024**3  # GB
        mem_peak = torch.cuda.max_memory_allocated() / 1024**3  # GB
        writer.add_scalar("system/gpu_mem_allocated_gb", mem_alloc, epoch)
        writer.add_scalar("system/gpu_mem_reserved_gb", mem_reserved, epoch)
        writer.add_scalar("system/gpu_mem_peak_gb", mem_peak, epoch)

    if epoch % cfg.val_vis_every == 0:
        t_val = time.time()
        model.eval()

        dice_metric.reset()
        train_patch_dice_per_class, train_patch_dice_mean = compute_patch_dice(
            model, train_loader, accelerator, dice_metric
        )
        dice_metric.reset()
        val_sw_dice_per_class, val_sw_dice_mean, val_sw_loss = compute_sw_dice(
            model, val_loader, accelerator, dice_metric, cfg.patch_size, cfg.sw_batch_size, cfg.sw_overlap
        )
        dice_metric.reset()
        val_time = time.time() - t_val
        val_times.append(val_time)

        logger.info(
            "Epoch %d train_dice=%.4f val_sw_dice=%.4f val_time=%.1fs",
            epoch, train_patch_dice_mean, val_sw_dice_mean, val_time
        )
        writer.add_scalar("train/dice_patch", float(train_patch_dice_mean), epoch)
        writer.add_scalar("val/dice_sw", float(val_sw_dice_mean), epoch)
        writer.add_scalar("val/loss_sw", float(val_sw_loss), epoch)
        writer.add_scalar("system/val_time_sec", val_time, epoch)

        for i, cls_name in enumerate(CLASSES):
            writer.add_scalar(f"train/dice_patch_{cls_name}", float(train_patch_dice_per_class[i].item()), epoch)
            writer.add_scalar(f"val/dice_sw_{cls_name}", float(val_sw_dice_per_class[i].item()), epoch)

        if val_sw_dice_mean > best_metric:
            best_metric = val_sw_dice_mean
            best_metric_epoch = epoch
            # 获取原始模型状态（unwrap accelerate wrapper）
            unwrapped_model = accelerator.unwrap_model(model)
            torch.save({
                "epoch": epoch,
                "model_state_dict": unwrapped_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_metric": best_metric,
                "dice_per_class": val_sw_dice_per_class.detach().cpu().numpy(),
            }, os.path.join(output_dir, "best.pt"))
            logger.info(f"Saved new best model -> {output_dir}/best.pt")

    unwrapped_model = accelerator.unwrap_model(model)
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": unwrapped_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_metric": best_metric,
        "best_metric_epoch": best_metric_epoch,
    }, os.path.join(output_dir, "last.pt"))

total_training_time = time.time() - training_start_time
avg_train_time = sum(train_times) / len(train_times) if train_times else 0
avg_val_time = sum(val_times) / len(val_times) if val_times else 0

logger.info("Training done. best_metric=%.4f at epoch=%d", best_metric, best_metric_epoch)
logger.info("Total time: %.1fs (%.2fh), avg_train=%.1fs, avg_val=%.1fs", 
            total_training_time, total_training_time / 3600, avg_train_time, avg_val_time)
writer.add_scalar("system/total_training_time_sec", total_training_time, cfg.epochs)
writer.add_scalar("system/avg_train_time_sec", avg_train_time, cfg.epochs)
writer.add_scalar("system/avg_val_time_sec", avg_val_time, cfg.epochs)
writer.close()

2026-02-07 03:49:25,226 | INFO | Mixed precision: fp16, Device: cuda
2026-02-07 03:49:25,228 | INFO | Scaler enabled: True
2026-02-07 03:49:25,228 | INFO | Output directory: outputs/20260207-034910_Unet3D_c1
/tmp/ipykernel_22989/2208631086.py:27: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  temp_scheduler.step()
2026-02-07 03:49:25,337 | INFO | Logged planned lr schedule (750 steps) to TensorBoard
2026-02-07 03:49:39,907 | INFO | Epoch 1 loss=1.0035 lr=2.13e-05 train_time=14.6s
2026-02-07 03:49:57,134 | INFO | Epoch 2 loss=1.0026 lr=2.53e-05 train_time=16.3s
2026-02-07 03:50:14,089 | INFO | Epoch 3 loss=1.0012 lr=3.19e-05 train_time=14.9s
2026-0

## Validation